# 01. AI가 내 연주를 듣다 — **기반**

## 이 노트북의 위치

```
기반 ✦   : NB01 (채보·분석) ← 여기
협업     : NB02 (Somax) + NB03 (RAVE)
확장     : NB04 (시각)
통합     : NB05 (무대)
```

이 노트북은 두 개의 피아노 연주를 AI로 분석하고, 같은 악기·같은 채보 도구가 **얼마나 다른 해석 세계**를 만들어내는지 체험합니다.

이 단계의 산출물(MIDI + 표현 분석)이 **확장·협업의 모든 기반**이 됩니다.

## 오늘의 두 연주

| | Satie — Gymnopédie No.1 | Prokofiev — Toccata Op.11 |
|---|---|---|
| 템포 | 느림 (~72 BPM) | 빠름 (~160+ BPM) |
| 텍스처 | 얇음 (1-3음) | 두꺼움 (6-10음) |
| 음역 | 좁음 (C3-C5) | 전체 건반 |
| 다이내믹 | pp-mp | f-fff |
| 어택 | 레가토 | 타격적 |

**도구**: [Basic Pitch](https://basicpitch.spotify.com/) (Spotify)

**출력**: `artifacts/midi/*.mid`, `artifacts/analysis/*.json` → 이후 모든 노트북에서 사용

In [ ]:
# Colab용 설치 (서버 시연 환경에서는 보통 건너뛰어도 됩니다)
!apt-get -qq install -y fluidsynth > /dev/null 2>&1
!pip install -q basic-pitch librosa soundfile midi2audio matplotlib pretty_midi numpy

import warnings
warnings.filterwarnings('ignore')
print('✅ 설치 완료!')

In [ ]:
# GitHub에서 프로젝트 클론 (assets 폴더 포함)
import os
if not os.path.exists('assets'):
    !git clone https://github.com/joonhyungbae/pianokit.git /tmp/pianokit
    !cp -r /tmp/pianokit/assets .
    print('✅ assets 폴더 준비 완료!')
else:
    print('✅ assets 폴더가 이미 존재합니다.')

---
## 1. 두 곡 로드 + 청취

먼저 두 연주를 들어봅니다. 청각적 대비가 이후 모든 분석의 근거가 됩니다.

In [ ]:
from pathlib import Path
import IPython.display as ipd

ASSETS = Path('assets')
ARTIFACTS = Path('artifacts')
(ARTIFACTS / 'midi').mkdir(parents=True, exist_ok=True)
(ARTIFACTS / 'analysis').mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'audio': ASSETS / 'satie_gymnopedie_no1.wav',
        'title': 'Satie — Gymnopédie No.1',
        'character': '서정형 (느림·얇음·좁은 음역)',
    },
    'prokofiev': {
        'audio': ASSETS / 'prokofiev_toccata_op11.wav',
        'title': 'Prokofiev — Toccata Op.11',
        'character': '동력형 (빠름·두꺼움·넓은 음역)',
    },
}

for key, p in pieces.items():
    if not p['audio'].exists():
        raise FileNotFoundError(f"{p['audio']} 를 찾을 수 없습니다. assets/ATTRIBUTIONS.md 참고.")
    print(f"🎵 {p['title']}")
    print(f"   {p['character']}")
    ipd.display(ipd.Audio(str(p['audio'])))
    print()

---
## 2. AI 채보 (Basic Pitch)

Spotify의 Basic Pitch가 두 연주를 동시에 MIDI로 변환합니다.
Satie는 수십 개 음표, Prokofiev는 천 단위 음표가 나올 것입니다.

In [ ]:
from basic_pitch.inference import predict

for key, p in pieces.items():
    print(f"🔄 {p['title']} 채보 중...")
    model_output, midi_data, note_events = predict(str(p['audio']))
    midi_path = ARTIFACTS / 'midi' / f'{key}.mid'
    midi_data.write(str(midi_path))
    p['midi_path'] = midi_path
    p['note_count'] = len(note_events)
    print(f"   ✅ {len(note_events)}개 음표 → {midi_path}")

print()
print('📊 음표 수 대비:')
print(f"   Satie:     {pieces['satie']['note_count']:>5}개")
print(f"   Prokofiev: {pieces['prokofiev']['note_count']:>5}개")
print(f"   배수: {pieces['prokofiev']['note_count'] / pieces['satie']['note_count']:.1f}x")

---
## 3. 피아노롤 비교

두 곡의 피아노롤을 나란히 놓고 비교합니다.
**음표 분포·밀도·음역대**가 시각적으로 완전히 다르게 드러납니다.

In [ ]:
import pretty_midi
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, (key, p) in zip(axes, pieces.items()):
    pm = pretty_midi.PrettyMIDI(str(p['midi_path']))
    for instrument in pm.instruments:
        for note in instrument.notes:
            ax.barh(note.pitch, note.end - note.start, left=note.start,
                    height=0.8, color=plt.cm.viridis(note.velocity / 127), alpha=0.7)
    all_pitches = [n.pitch for inst in pm.instruments for n in inst.notes]
    if all_pitches:
        p_min, p_max = min(all_pitches) - 2, max(all_pitches) + 2
        ax.set_ylim(p_min, p_max)
    ax.set_xlabel('시간 (초)')
    ax.set_ylabel('음높이 (MIDI)')
    ax.set_title(f"{p['title']} — {len(all_pitches)}음, 음역 {min(all_pitches)}-{max(all_pitches)}")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

---
## 4. 피아노 전용 표현 분석

일반 MIR 도구가 아니라 **피아노 연주가 관심 갖는 차원**으로 분석합니다.

- **velocity 곡선**: 시간별 다이내믹 (내 crescendo/decrescendo가 눈에 보임)
- **note density**: 초당 음표 수 (텍스처 밀도)
- **register spread**: 저음-고음 폭 (드라마의 크기)
- **IOI 분산 (CV)**: onset 간격의 요동 (루바토 지도)

### ⚠️ Basic Pitch 한계 공지

Basic Pitch는 오디오 → MIDI **채보**에는 강하지만 몇 가지 본질적 제약이 있습니다:

1. **Velocity 추정은 거친 근사**: 실제 피아노 다이내믹(pp~ff)을 정확히 복원하지 못합니다. 결과값은 보통 40~90 사이로 모여 있고, 절대값보다는 **같은 연주 내 상대 변화**로 읽어야 합니다.
2. **빠른 트릴·아르페지오**: 매우 빠른 연속음은 onset이 합쳐지거나 누락될 수 있습니다 (Prokofiev 16분음표에서 일부 나타남).
3. **페달로 뭉개진 저음**: 페달로 지속되는 저음 화음은 음량 감쇠 후에도 "울리는 음"으로 잡혀, 실제보다 density가 높게 추정될 수 있습니다.
4. **IOI CV 상승**: onset 검출의 미세 지터가 추가되어, 실제 루바토보다 CV 값이 약간 크게 나옵니다.

**결론**: 수치를 절대 기준이 아닌 **두 곡 비교의 상대 기준**으로 사용하세요. Satie vs Prokofiev의 대비는 이 한계에도 불구하고 명확히 드러납니다.

In [ ]:
import json

def analyze_performance(midi_path: Path) -> dict:
    """피아노 연주의 표현 차원을 추출."""
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = sorted(
        [n for inst in pm.instruments for n in inst.notes],
        key=lambda n: n.start
    )
    duration = pm.get_end_time()

    # 8초 구간별 프로파일
    seg_dur = 8.0
    segments = []
    for seg_start in np.arange(0, duration, seg_dur):
        seg_end = seg_start + seg_dur
        seg_notes = [n for n in notes if seg_start <= n.start < seg_end]
        if not seg_notes:
            continue
        vels = [n.velocity for n in seg_notes]
        pitches = [n.pitch for n in seg_notes]
        durs = [n.end - n.start for n in seg_notes]
        segments.append({
            'time_range': f'{seg_start:.1f}-{seg_end:.1f}s',
            'note_count': len(seg_notes),
            'density_per_sec': round(len(seg_notes) / seg_dur, 2),
            'avg_velocity': round(float(np.mean(vels)), 1),
            'velocity_range': f'{min(vels)}-{max(vels)}',
            'velocity_std': round(float(np.std(vels)), 1),
            'pitch_range': f'{pretty_midi.note_number_to_name(min(pitches))}-{pretty_midi.note_number_to_name(max(pitches))}',
            'register_spread': max(pitches) - min(pitches),
            'avg_duration': round(float(np.mean(durs)), 3),
        })

    # IOI (Inter-Onset Interval) 분산 = 루바토 지표
    iois = np.diff([n.start for n in notes])
    iois = iois[iois > 0]
    ioi_cv = float(np.std(iois) / np.mean(iois)) if len(iois) > 0 else 0.0

    # 전체 통계
    all_vels = [n.velocity for n in notes]
    all_pitches = [n.pitch for n in notes]

    return {
        'total_notes': len(notes),
        'duration_sec': round(duration, 2),
        'density_per_sec': round(len(notes) / duration, 2),
        'pitch_range': f'{pretty_midi.note_number_to_name(min(all_pitches))}-{pretty_midi.note_number_to_name(max(all_pitches))}',
        'register_spread_semitones': max(all_pitches) - min(all_pitches),
        'velocity_mean': round(float(np.mean(all_vels)), 1),
        'velocity_std': round(float(np.std(all_vels)), 1),
        'ioi_cv': round(ioi_cv, 3),  # 루바토 지표 (0=정확, >0.5=극단)
        'median_ioi': round(float(np.median(iois)), 4) if len(iois) > 0 else 0.0,
        'segments': segments,
    }


for key, p in pieces.items():
    analysis = analyze_performance(p['midi_path'])
    p['analysis'] = analysis
    out_path = ARTIFACTS / 'analysis' / f'{key}_analysis.json'
    out_path.write_text(json.dumps({'piece': p['title'], **analysis}, ensure_ascii=False, indent=2))
    print(f"✅ {p['title']}")
    print(f"   음표: {analysis['total_notes']}, 밀도: {analysis['density_per_sec']}/s, 음역: {analysis['register_spread_semitones']} 반음")
    print(f"   velocity: 평균 {analysis['velocity_mean']} (σ={analysis['velocity_std']})")
    print(f"   루바토 지표(IOI CV): {analysis['ioi_cv']} {'(극단)' if analysis['ioi_cv'] > 0.5 else '(안정)'}")
    print(f"   → {out_path}")
    print()

---
## 5. 비교 대시보드

두 연주의 **velocity 곡선·음표 밀도·음역 분포**를 나란히 놓습니다.

Satie는 완만한 파도, Prokofiev는 급격한 파동. 같은 피아노·같은 도구지만 **데이터가 완전히 다른 얼굴**을 보여줍니다.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
colors = {'satie': '#4f46e5', 'prokofiev': '#dc2626'}

for col, (key, p) in enumerate(pieces.items()):
    pm = pretty_midi.PrettyMIDI(str(p['midi_path']))
    notes = sorted([n for inst in pm.instruments for n in inst.notes], key=lambda n: n.start)
    c = colors[key]

    # (1) velocity 곡선
    times = [n.start for n in notes]
    vels = [n.velocity for n in notes]
    axes[0, col].scatter(times, vels, s=8, alpha=0.4, color=c)
    axes[0, col].set_title(f"{p['title']}\nvelocity 곡선")
    axes[0, col].set_ylabel('velocity')
    axes[0, col].set_ylim(0, 128)
    axes[0, col].grid(alpha=0.2)

    # (2) 밀도 (슬라이딩 윈도우 2초)
    dur = pm.get_end_time()
    t_grid = np.arange(0, dur, 0.2)
    density = np.zeros_like(t_grid)
    for i, t in enumerate(t_grid):
        density[i] = sum(1 for n in notes if t - 1.0 <= n.start < t + 1.0) / 2.0
    axes[1, col].fill_between(t_grid, density, alpha=0.6, color=c)
    axes[1, col].set_title('음표 밀도 (notes/sec)')
    axes[1, col].set_ylabel('density')
    axes[1, col].grid(alpha=0.2)

    # (3) 음역 히스토그램
    pitches = [n.pitch for n in notes]
    axes[2, col].hist(pitches, bins=30, color=c, alpha=0.7)
    axes[2, col].set_title('음역 분포')
    axes[2, col].set_xlabel('MIDI pitch')
    axes[2, col].set_ylabel('음표 수')
    axes[2, col].axvline(60, color='gray', linestyle='--', alpha=0.5, label='middle C')
    axes[2, col].legend()
    axes[2, col].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print('💡 관찰 포인트:')
print('   - velocity 폭: Satie는 좁은 밴드, Prokofiev는 위아래로 넓게 퍼짐')
print('   - 밀도 변화: Satie는 일정한 흐름, Prokofiev는 폭발적 구간')
print('   - 음역 분포: Satie는 중심 집중, Prokofiev는 양 극단')

---
## 다음 단계

여기서 추출한 MIDI와 분석 JSON은 다음 노트북에서 사용됩니다.

- `artifacts/midi/satie.mid`, `artifacts/midi/prokofiev.mid`
- `artifacts/analysis/satie_analysis.json`, `artifacts/analysis/prokofiev_analysis.json`

**→ NB04 (`04_visualize.ipynb`)**: 이 데이터로 시각 세계를 만듭니다.